In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("../data/raw/online_retail.csv")

# Display first 5 rows
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# Shape
print("Shape:", df.shape)

# Column names
print("\nColumns:\n", df.columns)

# Data types
print("\nData Types:\n", df.dtypes)

# Missing values
print("\nMissing Values:\n", df.isnull().sum())

Shape: (1067371, 8)

Columns:
 Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

Data Types:
 Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object

Missing Values:
 Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


In [4]:
df["Customer ID"].nunique()

5942

In [5]:
df[df["Quantity"] < 0].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


In [6]:
df[df["Invoice"].astype(str).str.startswith("C")].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


## Initial Data Observations
---

### Dataset Overview
- **Total Rows:** 1,067,371  
- **Total Columns:** 8  
- **Unique Customers:** 5,942  

### Data Quality Issues
- **Missing Customer IDs:** 243,007  
  → This may impact customer-level analysis.

- **Negative Quantities Present:**  
  → Indicates product returns.

- **Cancelled Invoices:**  
  → Invoice numbers starting with **"C"** represent cancellations.

### Next Step
Data cleaning is required before:
- Revenue analysis  
- Customer analysis  
- Segmentation

## 🧹 Data Cleaning
---

In [7]:
# Create a working copy
df_clean = df.copy()

print("Initial Shape:", df_clean.shape)

Initial Shape: (1067371, 8)


In [8]:
df_clean = df_clean[~df_clean["Invoice"].astype(str).str.startswith("C")]

print("After removing cancellations:", df_clean.shape)

After removing cancellations: (1047877, 8)


In [9]:
df_clean = df_clean[df_clean["Quantity"] > 0]

print("After removing negative quantities:", df_clean.shape)

After removing negative quantities: (1044420, 8)


In [10]:
df_clean = df_clean[df_clean["Customer ID"].notnull()]

print("After removing missing Customer IDs:", df_clean.shape)

After removing missing Customer IDs: (805620, 8)


In [14]:
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])
df_clean.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
Revenue               float64
dtype: object

In [13]:
#Revenue = Quantity × Price

df_clean["Revenue"] = df_clean["Quantity"] * df_clean["Price"]
df_clean.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [15]:
print("Final Cleaned Shape:", df_clean.shape)
print("Unique Customers:", df_clean["Customer ID"].nunique())
print("Total Revenue:", df_clean["Revenue"].sum())

Final Cleaned Shape: (805620, 9)
Unique Customers: 5881
Total Revenue: 17743429.178000007


# 🧹 Data Cleaning Summary
---
The following preprocessing steps were performed on the dataset:

- Removed cancelled invoices  
- Removed negative quantity transactions (returns)  
- Removed records with missing Customer IDs  
- Converted `InvoiceDate` to datetime format  
- Created a new column **Revenue** (`Quantity × Price`)  

The cleaned dataset is now ready for:

- 📊 Revenue analysis  
- 👥 Customer analysis  